In [ ]:
# 1. 营销任务优先级调度
class MarketingTask:
    """表示一个营销任务的类，包含名称、紧急程度和影响力属性。"""
    
    def __init__(self, name, urgency, impact):
        """
        初始化一个营销任务。
        """
        self.name = name
        self.urgency = urgency
        self.impact = impact
        self.priority = urgency * impact
    
    def update(self, urgency=None, impact=None):
        """
        更新任务的紧急程度和/或影响力。
        """
        if urgency is not None:
            self.urgency = urgency
        if impact is not None:
            self.impact = impact
        self.priority = self.urgency * self.impact


class MaxHeap:
    """用于按优先级存储营销任务的最大堆实现。"""
    
    def __init__(self):
        """初始化一个空的最大堆。"""
        self.heap = []
        self.size = 0
        self.position = {}  # 映射任务名称到其在堆中的位置
    
    def _parent(self, idx):
        """获取节点的父节点索引。"""
        return (idx - 1) // 2
    
    def _left_child(self, idx):
        """获取节点的左子节点索引。"""
        return 2 * idx + 1
    
    def _right_child(self, idx):
        """获取节点的右子节点索引。"""
        return 2 * idx + 2
    
    def _swap(self, i, j):
        """交换堆中的两个节点并更新它们的位置。"""
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]
        self.position[self.heap[i].name] = i
        self.position[self.heap[j].name] = j
    
    def _heapify_up(self, idx):
        """将节点向上移动到正确的位置。"""
        parent = self._parent(idx)
        if idx > 0 and self.heap[idx].priority > self.heap[parent].priority:
            self._swap(idx, parent)
            self._heapify_up(parent)
    
    def _heapify_down(self, idx):
        """将节点向下移动到正确的位置。"""
        largest = idx
        left = self._left_child(idx)
        right = self._right_child(idx)

        """与左右孩子中priority大的交换位置"""
        if left < self.size and self.heap[left].priority > self.heap[largest].priority:
            largest = left
        
        if right < self.size and self.heap[right].priority > self.heap[largest].priority:
            largest = right
        
        if largest != idx:
            self._swap(idx, largest)
            self._heapify_down(largest)
    
    def insert(self, task):
        if task.name in self.position:
            print(f"任务 '{task.name}' 已存在于堆中。")
            return
        
        self.heap.append(task)
        self.position[task.name] = self.size
        self.size += 1
        self._heapify_up(self.size - 1) #最后一个新插入的向上冒泡
    
    def get_max(self):
        """
        获取优先级最高的任务但不移除它。
        """
        if self.size == 0:
            return None
        return self.heap[0]
    
    def extract_max(self):
        """
        移除并返回优先级最高的任务。
        
        返回:
            MarketingTask: 优先级最高的任务
        """
        if self.size == 0:
            return None
        
        max_task = self.heap[0]
        """先交换，再冒泡"""
        self._swap(0, self.size - 1)
        del self.position[max_task.name]
        self.size -= 1
        self.heap.pop()
        
        if self.size > 0:
            self._heapify_down(0)
        
        return max_task
    
    def update_task(self, name, urgency=None, impact=None):
        if name not in self.position:
            print(f"任务 '{name}' 未在堆中找到。")
            return False
        
        """找到待更新的任务，则先更新priority"""
        idx = self.position[name]
        old_priority = self.heap[idx].priority
        self.heap[idx].update(urgency, impact)
        new_priority = self.heap[idx].priority
        
        """根据新老priority关系，选择向上/向下冒泡"""
        if new_priority > old_priority:
            self._heapify_up(idx)
        elif new_priority < old_priority:
            self._heapify_down(idx)
        
        return True
    
    def delete_task(self, name):
        if name not in self.position:
            print(f"任务 '{name}' 未在堆中找到。")
            return False
        
        """找到待删除的任务，则先交换再删除"""
        idx = self.position[name]
        self._swap(idx, self.size - 1)
        del self.position[name]
        self.size -= 1
        self.heap.pop()
        
        """交换上来的节点的priority只会小于等于原先被删除的节点，向下冒泡"""
        if idx < self.size:
            old_priority = self.heap[idx].priority
            self._heapify_down(idx)
            if self.heap[idx].priority == old_priority:
                self._heapify_up(idx)
        
        return True
    
    def get_top_k(self, k):
        if k <= 0:
            return []
        
        # 创建堆的副本
        temp_heap = self.heap[:self.size]
        temp_heap.sort(key=lambda x: x.priority, reverse=True)
        
        return temp_heap[:min(k, self.size)]

In [17]:
# 测试
# 创建最大堆实例
heap = MaxHeap()

# 插入任务
heap.insert(MarketingTask("任务A", urgency=5, impact=10))  # 优先级 50
heap.insert(MarketingTask("任务B", urgency=8, impact=6))   # 优先级 48
heap.insert(MarketingTask("任务C", urgency=7, impact=7))   # 优先级 49
heap.insert(MarketingTask("任务D", urgency=9, impact=5))   # 优先级 45

# 获取最大任务（优先级最高）
max_task = heap.get_max()
print(f"最大优先级任务: {max_task.name}，优先级: {max_task.priority}")

# 更新任务优先级
heap.update_task("任务D", urgency=10)  # 新优先级 10*5=50
print("更新任务D后，最大优先级任务:", heap.get_max().name)

# 删除任务
heap.delete_task("任务A")
print("删除任务A后，最大优先级任务:", heap.get_max().name)

# 提取最大任务（并移除）
extracted = heap.extract_max()
print(f"提取最大任务: {extracted.name}")

# 查询Top 2任务
top2 = heap.get_top_k(2)
print("Top 2任务:")
for task in top2:
    print(f"任务名称: {task.name}, 优先级: {task.priority}")

# 尝试插入已存在任务
heap.insert(MarketingTask("任务B", urgency=3, impact=3))  # 会提示已存在

# 尝试删除不存在任务
res = heap.delete_task("任务X")  # 会提示未找到
print("删除任务X结果:", res)


最大优先级任务: 任务A，优先级: 50
更新任务D后，最大优先级任务: 任务A
删除任务A后，最大优先级任务: 任务D
提取最大任务: 任务D
Top 2任务:
任务名称: 任务C, 优先级: 49
任务名称: 任务B, 优先级: 48
任务 '任务B' 已存在于堆中。
任务 '任务X' 未在堆中找到。
删除任务X结果: False


In [13]:
class Customer:
    """表示客户的类。"""
    def __init__(self, name):
        self.name = name

class CustomerNetwork:
    """表示客户关系的加权有向图的类。"""
    
    def __init__(self):
        """初始化一个空的客户网络。"""
        self.customers = {}  # 将客户名称映射到 Customer 对象
        self.graph = {}  # 将客户名称映射到邻居及权重的字典
    
    def add_customer(self, name):
        """加一个客户入网，其类别是一个类型为客户类的对象"""
        if name in self.customers:
            return self.customers[name]
        
        customer = Customer(name)
        self.customers[name] = customer
        self.graph[name] = {}
        return customer
    
    def add_relationship(self, from_name, to_name, weight):
        """添加有向关系，如果没这个客户，肯定先加上客户"""
        if from_name not in self.customers:
            self.add_customer(from_name)
        if to_name not in self.customers:
            self.add_customer(to_name)
        self.graph[from_name][to_name] = weight
    
    def remove_customer(self, name):
        if name not in self.customers:
            return False
        
        # 移除出边
        del self.graph[name]
        
        # 移除入边
        """检查每个用户的出边，如果有指向被移除用户的邻接，则删除"""
        for customer in self.graph:
            if name in self.graph[customer]:
                del self.graph[customer][name]
        
        del self.customers[name]
        return True
    
    def get_importance_scores(self):
        """
        每个客户的重要性 = 所有指向它的边的权重之和。
        """
        scores = {name: 0 for name in self.customers}
        
        for from_customer, neighbors in self.graph.items():
            for to_customer, weight in neighbors.items():
                scores[to_customer] += weight
        
        return scores
    
    def get_influenced_customers(self, name):
        """BFS广搜找所有被影响客户"""
        if name not in self.customers:
            return set()
        
        # 广度优先搜索
        visited = set()
        queue = [name]
        visited.add(name)
        
        """每次让基于current节点再走一步可以碰到的新节点入队"""
        while queue:
            current = queue.pop(0)
            for neighbor in self.graph[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
        
        # 从结果中移除源客户
        visited.remove(name)
        return visited

In [18]:
network = CustomerNetwork()

# 添加客户
network.add_customer("Alice")
network.add_customer("Bob")
network.add_customer("Charlie")
network.add_customer("David")

# 添加关系（有向加权边）
network.add_relationship("Alice", "Bob", 5)
network.add_relationship("Bob", "Charlie", 3)
network.add_relationship("Alice", "Charlie", 2)
network.add_relationship("Charlie", "David", 4)

# 打印重要性分数
scores = network.get_importance_scores()
print("重要性分数:")
for name, score in scores.items():
    print(f"{name}: {score}")

# 查找被影响客户（Alice影响的客户）
influenced = network.get_influenced_customers("Alice")
print(f"Alice影响的客户: {influenced}")

# 删除客户Bob
network.remove_customer("Bob")
print("删除Bob后，重要性分数:")
scores = network.get_importance_scores()
for name, score in scores.items():
    print(f"{name}: {score}")

# 再次查找Alice影响的客户
influenced = network.get_influenced_customers("Alice")
print(f"删除Bob后，Alice影响的客户: {influenced}")


重要性分数:
Alice: 0
Bob: 5
Charlie: 5
David: 4
Alice影响的客户: {'Bob', 'David', 'Charlie'}
删除Bob后，重要性分数:
Alice: 0
Charlie: 2
David: 4
删除Bob后，Alice影响的客户: {'David', 'Charlie'}


In [20]:
class Product:
    def __init__(self, product_id, name, price):
        self.product_id = product_id  # 商品唯一ID
        self.name = name              # 商品名称
        self.price = price            # 商品价格

# B+树节点类
class BPlusTreeNode:
    def __init__(self, is_leaf=False):
        self.is_leaf = is_leaf
        self.keys = []
        self.children = []
        self.next = None  # 新增：指向下一个叶子节点的指针，方便区间查询

# B+树类
class BPlusTree:
    def __init__(self, t=3):
        self.root = BPlusTreeNode(is_leaf=True)
        self.t = t  # 最小度数

    # 查找指定价格范围内的所有商品
    def search_range(self, low, high):
        result = []
        node = self._find_leaf(self.root, low)

        while node:
            for i, key in enumerate(node.keys):
                if low <= key <= high:
                    result.append(node.children[i])
                elif key > high:
                    return result
            # 遍历下一个叶子节点
            node = getattr(node, 'next', None)
        return result

    # 插入商品
    def insert(self, product: Product):
        root = self.root
        if len(root.keys) == (2 * self.t - 1):
            new_root = BPlusTreeNode()
            new_root.children.append(self.root)
            self._split_child(new_root, 0)
            self.root = new_root
        self._insert_non_full(self.root, product)

    # 删除key对应的商品
    def delete(self, price):
        node = self._find_leaf(self.root, price)
        if price in node.keys:
            idx = node.keys.index(price)
            node.keys.pop(idx)
            node.children.pop(idx)

    # 修改商品信息
    def update(self, price, new_product):
        node = self._find_leaf(self.root, price)
        if price in node.keys:
            idx = node.keys.index(price)
            node.children[idx] = new_product

    # 从当前节点往下找当前价格所在的叶子节点
    def _find_leaf(self, node, key):
        if node.is_leaf:
            return node
        # 如果当前节点不是叶子，遍历当前节点key对应的每一路
        for i, item in enumerate(node.keys):
            if key < item:
                return self._find_leaf(node.children[i], key)
        # 手动搜索最后一路（遍历不到）
        return self._find_leaf(node.children[-1], key)

    def _insert_non_full(self, node, product):
        # 对非满节点进行插入，key就是我的价格
        key = product.price
        #是叶子节点，直接找正确的地方插进去
        if node.is_leaf:
            i = 0
            #找到属于的那一路（处于左右两个价格中间）
            while i < len(node.keys) and key > node.keys[i]:
                i += 1
            #当前位置调用列表的插入
            node.keys.insert(i, key)
            node.children.insert(i, product)
        #非叶子节点，找正确的一路插进去
        else:
            i = 0
            while i < len(node.keys) and key > node.keys[i]:
                i += 1
            child = node.children[i]
            if len(child.keys) == (2 * self.t - 1):
                # 爆了，得分裂
                self._split_child(node, i)
                if key > node.keys[i]:
                    i += 1
            self._insert_non_full(node.children[i], product)

    def _split_child(self, parent, index):
        #分裂孩子
        t = self.t
        node = parent.children[index]
        new_node = BPlusTreeNode(is_leaf=node.is_leaf)

        parent.keys.insert(index, node.keys[t - 1])
        parent.children.insert(index + 1, new_node)

        new_node.keys = node.keys[t:]
        node.keys = node.keys[:t - 1]

        new_node.children = node.children[t:]
        node.children = node.children[:t]

        if node.is_leaf:
            # 如果是叶子节点，复制中间key到新节点，同时建立next指针
            new_node.keys = node.keys[t - 1:] + new_node.keys
            new_node.children = node.children[t - 1:] + new_node.children
            node.keys = node.keys[:t - 1]
            node.children = node.children[:t - 1]
            new_node.next = node.next
            node.next = new_node

In [ ]:
bpt = BPlusTree(t=3)

# 插入商品
products = [
    Product(1, "苹果", 10),
    Product(2, "香蕉", 5),
    Product(3, "橙子", 15),
    Product(4, "梨子", 20),
    Product(5, "葡萄", 12),
    Product(6, "西瓜", 25),
    Product(7, "草莓", 18),
]

for p in products:
    bpt.insert(p)

# 搜索价格区间 [10, 20]
results = bpt.search_range(10, 20)
print("价格区间 [10, 20] 内商品:")
for item in results:
    print(f"商品ID: {item.product_id}, 名称: {item.name}, 价格: {item.price}")

# 修改商品价格为15的商品名称
bpt.update(15, Product(3, "橙子（修改）", 15))
print("\n修改后价格15商品:")
results = bpt.search_range(15, 15)
for item in results:
    print(f"商品ID: {item.product_id}, 名称: {item.name}, 价格: {item.price}")

# 删除价格为5的商品
bpt.delete(5)
print("\n删除价格5商品后，区间 [0, 30] 内商品:")
results = bpt.search_range(0, 30)
for item in results:
    print(f"商品ID: {item.product_id}, 名称: {item.name}, 价格: {item.price}")


价格区间 [10, 20] 内商品:
商品ID: 1, 名称: 苹果, 价格: 10
商品ID: 5, 名称: 葡萄, 价格: 12
商品ID: 7, 名称: 草莓, 价格: 18
商品ID: 3, 名称: 橙子, 价格: 15

修改后价格15商品:
商品ID: 3, 名称: 橙子（修改）, 价格: 15

删除价格5商品后，区间 [0, 30] 内商品:
商品ID: 1, 名称: 苹果, 价格: 10
商品ID: 3, 名称: 橙子（修改）, 价格: 15
商品ID: 7, 名称: 草莓, 价格: 18
商品ID: 3, 名称: 橙子, 价格: 15
商品ID: 6, 名称: 西瓜, 价格: 25
